
# Sentiment Analysis

- Understand sentiment analysis.
- Prepare and clean text data.
- Convert text into numerical features using TF-IDF.
- Train a machine-learning sentiment classifier.
- Evaluate predictions with accuracy, precision, recall, F1-score and a confusion matrix.
- Predict sentiment for new sentences.

### Pipeline
```text
Raw Text → Cleaning → Train/Test Split → TF-IDF → ML Model → Prediction → Evaluation
```

## 1. What is Sentiment Analysis?

**Sentiment analysis** is an NLP task that identifies the opinion or polarity expressed in text.

| Text | Sentiment |
|---|---|
| I love this product. | Positive |
| This product is excellent. | Positive |
| I hate this service. | Negative |
| This service is terrible. | Negative |
| The package arrived today. | Neutral |

Common applications include product reviews, customer feedback, social media, surveys and support messages.

> A traditional ML sentiment model learns statistical patterns from labelled examples; it does not truly understand human emotions.

## 2. Import Libraries

We use Pandas for data, `re` for basic cleaning, and Scikit-learn for splitting, TF-IDF, classification and evaluation.

In [1]:
import re
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 3 Data Set

In [2]:
data = {
    "text": [
        "I love this product", "This product is excellent", "Amazing service and friendly staff",
        "The experience was wonderful", "I am very happy with the purchase", "The quality is fantastic",
        "This is a great application", "I really enjoyed the movie", "The support team was helpful", "Very good experience",
        "I hate this product", "This product is terrible", "The service was horrible", "I am very disappointed",
        "The quality is poor", "This is a bad experience", "I did not like the movie", "The support was useless",
        "Very frustrating experience", "I regret buying this product",
        "The package arrived today", "The product is black", "The meeting starts at ten", "I received the order yesterday",
        "The laptop has 16 GB RAM", "The store is open today", "The course has four modules",
        "The phone was delivered in the morning", "The application was updated", "The class starts at nine"
    ],
    "sentiment": (
        ["positive"] * 10 + ["negative"] * 10 + ["neutral"] * 10
    )
}

df = pd.DataFrame(data)
df.head()

,text,sentiment
0,I love this product,positive
1,This product is excellent,positive
2,Amazing service and friendly staff,positive
3,The experience was wonderful,positive
4,I am very happy with the purchase,positive


## 4. Inspect the Dataset

In [3]:
print("Dataset shape:", df.shape)
print("\nClass counts:")
print(df["sentiment"].value_counts())

Dataset shape: (30, 2)

Class counts:
sentiment
positive    10
negative    10
neutral     10
Name: count, dtype: int64


In [4]:
df.sample(5,random_state=42)

,text,sentiment
27,The phone was delivered in the morning,neutral
15,This is a bad experience,negative
23,I received the order yesterday,neutral
17,The support was useless,negative
8,The support team was helpful,positive


## 5. Basic Text Cleaning

Real text can contain capital letters, punctuation, URLs, extra spaces and special characters. This simple function normalizes some of them.

**Important:** preprocessing is task-dependent. Do not blindly remove information such as punctuation or emojis if it carries useful sentiment signals.

In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text"]].head(10)

,text,clean_text
0,I love this product,i love this product
1,This product is excellent,this product is excellent
2,Amazing service and friendly staff,amazing service and friendly staff
3,The experience was wonderful,the experience was wonderful
4,I am very happy with the purchase,i am very happy with the purchase
5,The quality is fantastic,the quality is fantastic
6,This is a great application,this is a great application
7,I really enjoyed the movie,i really enjoyed the movie
8,The support team was helpful,the support team was helpful
9,Very good experience,very good experience


## 6. Separate Features and Target

`X` contains the text. `y` contains the sentiment label.

In [6]:
X = df["clean_text"]
y = df["sentiment"]
print("Examples:", len(X))
print("Classes:", y.unique())

Examples: 30
Classes: <StringArray>
['positive', 'negative', 'neutral']
Length: 3, dtype: str


## 7. Train-Test Split

The training set is used to learn patterns. The test set is kept unseen until evaluation. `stratify=y` helps preserve class proportions.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training examples:", len(X_train))
print("Testing examples :", len(X_test))

Training examples: 22
Testing examples : 8


## 8. TF-IDF: Convert Text into Numbers

Traditional ML algorithms need numerical input. **TF-IDF (Term Frequency–Inverse Document Frequency)** gives statistical importance to terms.

A word that occurs often in one document but is not common across many documents can receive a larger weight. Very common terms receive less distinguishing power.

We use `ngram_range=(1, 2)` so the model can use both individual words and two-word phrases such as `very good` and `not good`.

In [8]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training matrix shape:", X_train_tfidf.shape)
print("Testing matrix shape :", X_test_tfidf.shape)

Training matrix shape: (22, 116)
Testing matrix shape : (8, 116)


### Why `fit_transform()` for training and only `transform()` for test?

`fit_transform(X_train)` learns vocabulary/IDF information from training data and transforms it.

`transform(X_test)` uses the already learned representation. We should not fit a separate vectorizer on test data because that would leak information from the evaluation set into the modelling process.

## 9. Inspect the Vocabulary

In [9]:
features = vectorizer.get_feature_names_out()
print("Number of features:", len(features))
print(features[:50])

Number of features: 116
['am' 'am very' 'amazing' 'amazing service' 'and' 'and friendly'
 'application' 'application was' 'arrived' 'arrived today' 'bad'
 'bad experience' 'black' 'buying' 'buying this' 'course' 'course has'
 'did' 'did not' 'disappointed' 'enjoyed' 'enjoyed the' 'experience'
 'experience was' 'fantastic' 'four' 'four modules' 'friendly'
 'friendly staff' 'frustrating' 'frustrating experience' 'gb' 'gb ram'
 'good' 'good experience' 'great' 'great application' 'has' 'has four'
 'has gb' 'hate' 'hate this' 'helpful' 'horrible' 'is' 'is bad' 'is black'
 'is fantastic' 'is great' 'is open']


## 10. Train Logistic Regression

Logistic Regression is a simple and effective baseline for text classification. It learns relationships between TF-IDF features and sentiment labels.

In [10]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)
print("Model training completed.")

Model training completed.


## 11. Make Predictions

In [11]:
y_pred = model.predict(X_test_tfidf)

results = pd.DataFrame({
    "Text": X_test.values,
    "Actual": y_test.values,
    "Predicted": y_pred
})

print(results)

                                     Text    Actual Predicted
0                     the quality is poor  negative  positive
1       i am very happy with the purchase  positive  negative
2               this product is excellent  positive  negative
3               the meeting starts at ten   neutral   neutral
4                     i love this product  positive  negative
5                the class starts at nine   neutral   neutral
6                 the support was useless  negative  positive
7  the phone was delivered in the morning   neutral   neutral


## 12. Accuracy

Accuracy is the proportion of correct predictions among all predictions.

```text
Accuracy = Correct Predictions / Total Predictions
```

In [12]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 37.50%


## 13. Classification Report

The report provides **precision, recall, F1-score and support** for each class.

- **Precision:** among predictions for a class, how many were correct?
- **Recall:** among actual examples of a class, how many were found?
- **F1-score:** combines precision and recall.
- **Support:** number of actual examples in that class.

In [13]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         2
     neutral       1.00      1.00      1.00         3
    positive       0.00      0.00      0.00         3

    accuracy                           0.38         8
   macro avg       0.33      0.33      0.33         8
weighted avg       0.38      0.38      0.38         8



## 14. Confusion Matrix

A confusion matrix shows where predictions were correct and which classes were confused with each other.

In [14]:
labels = ["positive", "negative", "neutral"]
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"Actual {x}" for x in labels], columns=[f"Predicted {x}" for x in labels])
cm_df

,Predicted positive,Predicted negative,Predicted neutral
Actual positive,0,3,0
Actual negative,2,0,0
Actual neutral,0,0,3


## 15. Predict New Sentences

Now we use the trained pipeline on new text that was not in the training dataset.

In [15]:
new_text = [
    "I really love this service",
    "This is a terrible product",
    "The package arrived this morning",
    "The application is fantastic",
    "I am disappointed with the service"
]

new_clean = [clean_text(text) for text in new_text]
new_tfidf = vectorizer.transform(new_clean)
predictions = model.predict(new_tfidf)

for text, prediction in zip(new_text, predictions):
    print(f"{text:45} → {prediction}")

I really love this service                    → negative
This is a terrible product                    → negative
The package arrived this morning              → neutral
The application is fantastic                  → positive
I am disappointed with the service            → negative
